In [ ]:
# Set environment variables — must run this FIRST before starting Spark
import os

# Java 11 (compatible with PySpark 3.5.x)
os.environ['JAVA_HOME'] = r'C:\Program Files\Eclipse Adoptium\jdk-11.0.28.6-hotspot'
os.environ['PATH'] = r'C:\Program Files\Eclipse Adoptium\jdk-11.0.28.6-hotspot\bin' + ';' + os.environ['PATH']

# Hadoop winutils — required for PySpark to work on Windows
os.environ['HADOOP_HOME'] = r'C:\hadoop'
os.environ['PATH'] = r'C:\hadoop\bin' + ';' + os.environ['PATH']

# Tell Spark's JVM which Python executable to use for worker processes
# On Windows the binary is python.exe not python3
os.environ['PYSPARK_PYTHON'] = r'C:\Users\advai\anaconda3\envs\pyspark_env\python.exe'
os.environ['PYSPARK_DRIVER_PYTHON'] = r'C:\Users\advai\anaconda3\envs\pyspark_env\python.exe'

# Remove any external SPARK_HOME so pyspark uses its own bundled JARs (3.5.3)
os.environ.pop('SPARK_HOME', None)

import pyspark
print('JAVA_HOME:', os.environ['JAVA_HOME'])
print('HADOOP_HOME:', os.environ['HADOOP_HOME'])
print('PYSPARK_PYTHON:', os.environ['PYSPARK_PYTHON'])

In [2]:
import sys
print(sys.executable)

C:\Users\advai\anaconda3\envs\pyspark_env\python.exe


# Framework Local Testing Notebook

Tests the pure utility functions in `framework.py` using a local PySpark session.
No cloud credentials, Databricks, or Snowflake required.

**Functions covered:**
- Global Utilities: column ops, type casting, MD5 checksums, DML helpers
- Informatica-style Functions: router, joiner, checksum, insert/update/delete flags, lookup
- Regression Testing: schema comparison, hash comparison, dataframe diff, special character check

**Setup:**
```bash
pip install pyspark pandas
```

## 1. Start a Local Spark Session

In [ ]:
# display() helper — mimics Databricks display(), renders as HTML table in Jupyter
def display(df, n=20):
    """Show a PySpark DataFrame as a formatted HTML table (like Databricks display)."""
    return df.limit(n).toPandas()

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("framework-local-test") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")  # suppress INFO/WARN noise
print(f"Spark version: {spark.version}")

Spark version: 3.5.3


## 2. Import Framework

The module-level `SparkSession` and `dbutils` lines are commented out in `framework.py`,
so the import will succeed locally. Functions that reference `spark` or `dbutils` at
call-time will use the session created above once we inject it.

In [4]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))  # ensure framework.py is on the path

import importlib
import framework

# Inject the local spark session so framework functions can use it
framework.spark = spark

# Expose all public names into this namespace for convenience
from framework import (
    GetValueFromDataframe, GetTime, julian_to_timestamp, epoch_to_datetime,
    col_rename, truncate_dataframe, column_clear, uppercase_columns,
    trim_column_values, find_and_replace, column_retitle, df_column_rename,
    add_column_prefix, add_prefix_suffix, to_string_datatype,
    generate_update_setString, remove_null_from_dictionary,
    find_and_replace_within_values, dml_operation_df, calculate_df_size,
    infa_router, infa_joiner, md5_checksum,
    generate_insert_update_delete_flags, generate_insert_update_delete_dataframes,
    lookup_dataframe, compare_schemas, hash_comp, compare_dataframes,
    check_for_special_characters, generate_delta_merge_conditions
)
print("Import successful.")

Import successful.


## 3. Sample DataFrames

Reusable test DataFrames used throughout the notebook.

In [5]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Source DataFrame — simulates a raw ingestion payload
source_data = [
    Row(id=1, first_name="Alice",  last_name="Smith",  salary=50000.0, dept="Engineering"),
    Row(id=2, first_name="Bob",    last_name="Jones",  salary=60000.0, dept="Finance"),
    Row(id=3, first_name="Carol",  last_name="White",  salary=55000.0, dept="Engineering"),
    Row(id=4, first_name="  Dave", last_name="Brown ", salary=None,    dept="HR"),
    Row(id=5, first_name="Eve",    last_name="Davis",  salary=70000.0, dept="Finance"),
]

source_schema = StructType([
    StructField("id",         IntegerType(), True),
    StructField("first_name", StringType(),  True),
    StructField("last_name",  StringType(),  True),
    StructField("salary",     DoubleType(),  True),
    StructField("dept",       StringType(),  True),
])

source_df = spark.createDataFrame(source_data, schema=source_schema)

# Target DataFrame — simulates existing data in the warehouse
target_data = [
    Row(id=1, first_name="Alice", last_name="Smith", salary=48000.0, dept="Engineering"),  # salary changed
    Row(id=2, first_name="Bob",   last_name="Jones", salary=60000.0, dept="Finance"),      # unchanged
    Row(id=6, first_name="Frank", last_name="Green", salary=45000.0, dept="Ops"),          # delete candidate
]
target_df = spark.createDataFrame(target_data, schema=source_schema)

print("Source DataFrame:")
display(source_df)
print("Target DataFrame:")
display(target_df)

Source DataFrame:


Py4JJavaError: An error occurred while calling o41.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 0.0 failed 1 times, most recent failure: Lost task 0.0 in stage 0.0 (TID 0) (LegionAK.mshome.net executor driver): java.io.IOException: Cannot run program "python3": CreateProcess error=2, The system cannot find the file specified
	at java.base/java.lang.ProcessBuilder.start(ProcessBuilder.java:1128)
	at java.base/java.lang.ProcessBuilder.start(ProcessBuilder.java:1071)
	at org.apache.spark.api.python.PythonWorkerFactory.createSimpleWorker(PythonWorkerFactory.scala:181)
	at org.apache.spark.api.python.PythonWorkerFactory.create(PythonWorkerFactory.scala:109)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:124)
	at org.apache.spark.api.python.BasePythonRunner.compute(PythonRunner.scala:174)
	at org.apache.spark.api.python.PythonRDD.compute(PythonRDD.scala:67)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoint(RDD.sc

## 4. Global Utilities

### 4.1 GetTime

In [ ]:
print("Current timestamp (default):", GetTime())
print("Current timestamp (custom fmt):", GetTime(strform='%Y-%m-%d %H:%M:%S'))

### 4.2 col_rename — rename column dictionary

In [ ]:
rename_map = {"first_name": "FirstName", "last_name": "LastName", "salary": "AnnualSalary"}
renamed_df = col_rename(source_df, rename_map)
display(renamed_df)

### 4.3 uppercase_columns

In [ ]:
upper_df = uppercase_columns(source_df, ['first_name', 'last_name'])
display(upper_df)

### 4.4 trim_column_values — strip leading/trailing whitespace

In [ ]:
trimmed_df = trim_column_values(source_df)
# Row 4 (Dave / Brown) had leading/trailing spaces — should be clean now
trimmed_df.filter(trimmed_df.id == 4).show()

### 4.5 add_column_prefix / add_prefix_suffix

In [ ]:
prefixed_df = add_column_prefix(source_df, prefix='src_', columns=['first_name', 'last_name'])
print(prefixed_df.schema.simpleString())

### 4.6 to_string_datatype — cast all columns to StringType

In [ ]:
str_df = to_string_datatype(source_df)
print(str_df.schema.simpleString())
display(str_df)

### 4.7 calculate_df_size

In [ ]:
size_bytes = calculate_df_size(source_df)
print(f"Estimated DataFrame size: {size_bytes} bytes")

### 4.8 GetValueFromDataframe

In [ ]:
# Returns first value of a column from the first row
val = GetValueFromDataframe(source_df, 'first_name')
print("First value of first_name:", val)

### 4.9 find_and_replace

In [ ]:
replaced_df = find_and_replace(source_df, 'dept', 'Engineering', 'Tech')
display(replaced_df)

### 4.10 truncate_dataframe — limit to N rows

In [ ]:
trunc_df = truncate_dataframe(source_df, 3)
print("Row count after truncate:", trunc_df.count())
display(trunc_df)

### 4.11 generate_update_setString

In [ ]:
set_str = generate_update_setString(source_df, exclude_cols=['id'])
print("UPDATE SET clause:", set_str)

### 4.12 generate_delta_merge_conditions

In [ ]:
merge_cond = generate_delta_merge_conditions(source_df, pk_columns=['id'])
print("MERGE ON condition:", merge_cond)

## 5. Informatica-Style Functions

### 5.1 infa_router — split DataFrame by condition

In [ ]:
# Route into Engineering vs. everything else
routes = {
    "engineering": "dept = 'Engineering'",
    "other":       "dept != 'Engineering'",
}
routed = infa_router(source_df, routes)
print("Engineering group:")
routed['engineering'].show()
print("Other group:")
routed['other'].show()

### 5.2 infa_joiner

In [ ]:
dept_data = spark.createDataFrame([
    Row(dept="Engineering", budget=500000),
    Row(dept="Finance",     budget=300000),
    Row(dept="HR",          budget=150000),
])

joined_df = infa_joiner(source_df, dept_data, join_key='dept', join_type='left')
display(joined_df)

### 5.3 md5_checksum

In [ ]:
checksum_df = md5_checksum(source_df, columns=['first_name', 'last_name', 'salary'], checksum_col='row_hash')
checksum_df.select('id', 'first_name', 'row_hash').show(truncate=False)

### 5.4 generate_insert_update_delete_flags

Compares source and target DataFrames and tags each row with I (insert), U (update), or D (delete).

In [ ]:
flagged_df = generate_insert_update_delete_flags(
    source_df=source_df,
    target_df=target_df,
    pk_columns=['id']
)
display(flagged_df)

### 5.5 generate_insert_update_delete_dataframes

Returns a dict of `{"insert": df, "update": df, "delete": df}`.

In [ ]:
dml_dfs = generate_insert_update_delete_dataframes(
    source_df=source_df,
    target_df=target_df,
    pk_columns=['id']
)

print(f"Inserts: {dml_dfs['insert'].count()}")
dml_dfs['insert'].show()

print(f"Updates: {dml_dfs['update'].count()}")
dml_dfs['update'].show()

print(f"Deletes: {dml_dfs['delete'].count()}")
dml_dfs['delete'].show()

### 5.6 lookup_dataframe

In [ ]:
lookup_ref = spark.createDataFrame([
    Row(dept="Engineering", dept_code="ENG"),
    Row(dept="Finance",     dept_code="FIN"),
    Row(dept="HR",          dept_code="HR"),
])

enriched_df = lookup_dataframe(
    source_df=source_df,
    lookup_df=lookup_ref,
    lookup_key='dept',
    lookup_columns=['dept_code']
)
display(enriched_df)

## 6. Regression Testing Functions

### 6.1 compare_schemas

In [ ]:
# Same schema — expect no differences
result = compare_schemas(source_df, target_df)
print("Schema comparison result:", result)

# Add an extra column to one DF to trigger a mismatch
from pyspark.sql.functions import lit
modified_df = source_df.withColumn("extra_col", lit("test"))
result_mismatch = compare_schemas(source_df, modified_df)
print("Schema mismatch result:", result_mismatch)

### 6.2 hash_comp — row-level hash comparison

In [ ]:
hash_result = hash_comp(source_df, target_df, pk_columns=['id'])
print("Hash comparison result:")
display(hash_result)

### 6.3 compare_dataframes — full diff

In [ ]:
diff_result = compare_dataframes(source_df, target_df)
print("DataFrame diff:")
display(diff_result)

### 6.4 check_for_special_characters

In [ ]:
# Add some special characters to test
special_data = spark.createDataFrame([
    Row(id=1, name="Alice & Bob"),
    Row(id=2, name="Carol <normal>"),
    Row(id=3, name="Dave"),
])

spec_result = check_for_special_characters(special_data, columns=['name'])
display(spec_result)

## 7. DML Operation Helper — dml_operation_df

Builds an insert/update/delete DML statement string from a DataFrame row.

In [ ]:
dml_str = dml_operation_df(
    df=source_df,
    table_name='employees',
    operation='insert',
    pk_columns=['id']
)
print("Generated DML:")
print(dml_str)

## 8. Utility Helpers

### 8.1 julian_to_timestamp / epoch_to_datetime

In [ ]:
julian_df = spark.createDataFrame([Row(jdate=2460000)])
ts_df = julian_to_timestamp(julian_df, julian_col='jdate')
display(ts_df)

epoch_df = spark.createDataFrame([Row(epoch_ms=1700000000000)])
dt_df = epoch_to_datetime(epoch_df, epoch_col='epoch_ms')
display(dt_df)

### 8.2 remove_null_from_dictionary

In [ ]:
d = {"a": 1, "b": None, "c": "hello", "d": None}
clean_d = remove_null_from_dictionary(d)
print("Cleaned dict:", clean_d)

### 8.3 column_clear — fill column with a default value

In [ ]:
cleared_df = column_clear(source_df, columns=['salary'], fill_value=0.0)
display(cleared_df)

---
## Summary

| Section | Functions Tested |
|---------|------------------|
| Global Utilities | GetTime, col_rename, uppercase_columns, trim_column_values, add_column_prefix, to_string_datatype, calculate_df_size, GetValueFromDataframe, find_and_replace, truncate_dataframe, generate_update_setString, generate_delta_merge_conditions |
| Informatica Functions | infa_router, infa_joiner, md5_checksum, generate_insert_update_delete_flags, generate_insert_update_delete_dataframes, lookup_dataframe |
| Regression Testing | compare_schemas, hash_comp, compare_dataframes, check_for_special_characters |
| DML Helpers | dml_operation_df |
| Utilities | julian_to_timestamp, epoch_to_datetime, remove_null_from_dictionary, column_clear |

For cloud-dependent functions (blob, snowflake, on-prem JDBC, Salesforce, REST API),
refer to `notes/framework_progress.md` for the setup roadmap.